## 1. 라이브러리 및 환경 설정

In [40]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from catboost import CatBoostRegressor
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## 2. 데이터 로드

In [41]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

train = train.merge(layout, on='layout_id', how='left')
test = test.merge(layout, on='layout_id', how='left')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")
# print(f"학습 head : {train.head}")

학습 데이터 크기: (250000, 108)
테스트 데이터 크기: (50000, 107)


### 기본/비율/곱 파생변수 + time_idx

In [42]:
# layout_type 처리
combined = pd.concat([train['layout_type'], test['layout_type']], axis=0).astype(str)
mapping = {v: i for i, v in enumerate(combined.unique())}
train['layout_type'] = train['layout_type'].astype(str).map(mapping)
test['layout_type'] = test['layout_type'].astype(str).map(mapping)

# ===== 기본 비율 feature =====
train['order_per_robot'] = train['order_inflow_15m'] / (train['robot_active'] + 1)
test['order_per_robot'] = test['order_inflow_15m'] / (test['robot_active'] + 1)

train['charge_pressure'] = train['robot_charging'] / (train['charger_count'] + 1)
test['charge_pressure'] = test['robot_charging'] / (test['charger_count'] + 1)

train['congestion_per_robot'] = train['congestion_score'] / (train['robot_active'] + 1)
test['congestion_per_robot'] = test['congestion_score'] / (test['robot_active'] + 1)

# ===== 기본 파생 변수 =====
train['order_per_robot'] = train['order_inflow_15m'] / (train['robot_active'] + 1)
test['order_per_robot'] = test['order_inflow_15m'] / (test['robot_active'] + 1)

train['charge_pressure'] = train['robot_charging'] / (train['charger_count'] + 1)
test['charge_pressure'] = test['robot_charging'] / (test['charger_count'] + 1)

train['congestion_per_robot'] = train['congestion_score'] / (train['robot_active'] + 1)
test['congestion_per_robot'] = test['congestion_score'] / (test['robot_active'] + 1)

# ===== 운영 비율 / 병목 feature 확장 =====
train['orders_per_pack_station'] = train['order_inflow_15m'] / (train['pack_station_count'] + 1)
test['orders_per_pack_station'] = test['order_inflow_15m'] / (test['pack_station_count'] + 1)

train['active_per_aisle_width'] = train['robot_active'] / (train['aisle_width_avg'] + 1)
test['active_per_aisle_width'] = test['robot_active'] / (test['aisle_width_avg'] + 1)

train['congestion_per_aisle'] = train['congestion_score'] / (train['aisle_width_avg'] + 1)
test['congestion_per_aisle'] = test['congestion_score'] / (test['aisle_width_avg'] + 1)

train['charging_per_robot'] = train['robot_charging'] / (train['robot_active'] + 1)
test['charging_per_robot'] = test['robot_charging'] / (test['robot_active'] + 1)

train['low_battery_pressure'] = train['low_battery_ratio'] * train['robot_active']
test['low_battery_pressure'] = test['low_battery_ratio'] * test['robot_active']

train['pack_pressure'] = train['pack_utilization'] / (train['pack_station_count'] + 1)
test['pack_pressure'] = test['pack_utilization'] / (test['pack_station_count'] + 1)

train['intersection_pressure'] = train['congestion_score'] * train['intersection_count']
test['intersection_pressure'] = test['congestion_score'] * test['intersection_count']

train['robot_density'] = train['robot_total'] / (train['floor_area_sqm'] + 1)
test['robot_density'] = test['robot_total'] / (test['floor_area_sqm'] + 1)

train['charger_per_robot'] = train['charger_count'] / (train['robot_total'] + 1)
test['charger_per_robot'] = test['charger_count'] / (test['robot_total'] + 1)

train['emergency_complexity'] = train['emergency_exit_count'] / (train['floor_area_sqm'] + 1)
test['emergency_complexity'] = test['emergency_exit_count'] / (test['floor_area_sqm'] + 1)

# ===== interaction feature (곱) =====
train['congestion_x_robot_active'] = train['congestion_score'] * train['robot_active']
test['congestion_x_robot_active'] = test['congestion_score'] * test['robot_active']

train['order_inflow_x_congestion'] = train['order_inflow_15m'] * train['congestion_score']
test['order_inflow_x_congestion'] = test['order_inflow_15m'] * test['congestion_score']

train['low_battery_x_robot_active'] = train['low_battery_ratio'] * train['robot_active']
test['low_battery_x_robot_active'] = test['low_battery_ratio'] * test['robot_active']

train['pack_util_x_order_inflow'] = train['pack_utilization'] * train['order_inflow_15m']
test['pack_util_x_order_inflow'] = test['pack_utilization'] * test['order_inflow_15m']

# ===== pseudo time feature =====
train['time_idx'] = train.groupby('scenario_id').cumcount()
test['time_idx'] = test.groupby('scenario_id').cumcount()

train = train.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
test = test.sort_values(['scenario_id', 'ID']).reset_index(drop=True)

# train/test 동일 기준으로 layout_type 인코딩
combined = pd.concat([train['layout_type'], test['layout_type']], axis=0).astype(str)
mapping = {v: i for i, v in enumerate(combined.unique())}

train['layout_type'] = train['layout_type'].astype(str).map(mapping)
test['layout_type'] = test['layout_type'].astype(str).map(mapping)

print(train['layout_type'].head())
print(train['layout_type'].dtype)

0    0
1    0
2    0
3    0
4    0
Name: layout_type, dtype: int64
int64


### 결측 feature + 시계열 feature

In [43]:
# ===== 결측 feature =====
missing_ratio = train.isnull().mean().sort_values(ascending=False)
top_missing_cols = missing_ratio[missing_ratio > 0].head(10).index.tolist()

print("결측 컬럼:", top_missing_cols)

for col in top_missing_cols:
    train[f'{col}_isna'] = train[col].isna().astype(int)
    test[f'{col}_isna'] = test[col].isna().astype(int)

train['num_missing'] = train.isnull().sum(axis=1)
test['num_missing'] = test.isnull().sum(axis=1)


def add_time_features(df):
    df = df.copy()

    group_col = 'scenario_id'

    # lag / diff용
    base_cols = [
        'order_inflow_15m',
        'congestion_score',
        'robot_charging',
        'battery_mean',
        'pack_utilization'
    ]

    for col in base_cols:
        # lag
        df[f'{col}_lag1'] = df.groupby(group_col)[col].shift(1)
        df[f'{col}_lag2'] = df.groupby(group_col)[col].shift(2)

        # diff
        df[f'{col}_diff1'] = df[col] - df.groupby(group_col)[col].shift(1)

    # ===== rolling mean / std 추가 =====
    rolling_mean_cols = [
        'order_inflow_15m',
        'congestion_score',
        'robot_charging'
    ]

    for col in rolling_mean_cols:
        df[f'{col}_rolling_mean_2'] = (
            df.groupby(group_col)[col]
              .transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())
        )

        df[f'{col}_rolling_mean_3'] = (
            df.groupby(group_col)[col]
              .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        )

    # battery_mean은 std까지 추가
    df['battery_mean_rolling_std_3'] = (
        df.groupby(group_col)['battery_mean']
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).std())
    )

    return df

train = add_time_features(train)
test = add_time_features(test)

train = train.fillna(0)
test = test.fillna(0)

결측 컬럼: ['order_inflow_x_congestion', 'pack_util_x_order_inflow', 'avg_recovery_time', 'congestion_per_robot', 'congestion_per_aisle', 'congestion_score', 'congestion_x_robot_active', 'intersection_pressure', 'avg_charge_wait', 'battery_mean']


### 타깃/피처 정의

In [46]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
# 학습 전용 보조 타깃 (피처에서 제외)
TRAIN_ONLY_AUX = ['_total_delay_proxy', '_delay_level']

y_series = train[TARGET]
# avg ≈ total / volume → proxy: 15분 유입 주문량을 출고량 proxy로 사용
train['_order_vol_proxy'] = np.maximum(train['order_inflow_15m'].fillna(0).values, 1.0)
test['_order_vol_proxy'] = np.maximum(test['order_inflow_15m'].fillna(0).values, 1.0)
train['_total_delay_proxy'] = y_series.values * train['_order_vol_proxy']
# delay_level (low/mid/high) — tertile
q33, q66 = y_series.quantile([0.33, 0.66])
train['_delay_level'] = pd.cut(
    y_series, bins=[-np.inf, q33, q66, np.inf], labels=[0, 1, 2]
).astype(int)

# layout 기반 interaction (로컬 패턴 보강)
for d in (train, test):
    lt = d['layout_type'].astype(float)
    d['layout_x_congestion'] = lt * d['congestion_score']
    d['layout_x_order_inflow'] = lt * d['order_inflow_15m'].fillna(0)
    d['layout_x_robot_active'] = lt * d['robot_active']

feature_cols = [
    c for c in train.columns
    if c not in ID_COLS + [TARGET] + TRAIN_ONLY_AUX + ['_order_vol_proxy']
]
seq_feature_cols = feature_cols.copy()

TOP_K_FEATURES = 80
# 분해 브랜치: 출고량 proxy를 피처에서 빼야 총지연/주문량을 따로 학습하는 의미가 있음
DECOMP_DROP = {'order_inflow_15m', '_order_vol_proxy'}

print(f"원본 feature 수: {len(feature_cols)} (TOP_K={TOP_K_FEATURES} 으로 축소 예정)")

sequence feature 수: 155


### 탭형 입력 변수

In [47]:
y = train[TARGET]
groups = train['scenario_id']

X_full = train[feature_cols]
X_test_full = test[feature_cols]

# sample weight: 큰 지연 강조 + log 스케일
sw = (1.0 + np.log1p(y.values)) * np.where(y.values > 30, 2.0, 1.0)


def select_top_features_lgb(train_df, y, groups, cols, top_n, sample_weight=None, n_folds_importance=3, seed=42):
    """LightGBM gain 기준 상위 top_n 피처 (과적합·노이즈 완화)."""
    kf = GroupKFold(n_splits=5)
    acc = np.zeros(len(cols))
    X = train_df[cols]
    n_used = 0
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y, groups=groups)):
        if fold >= n_folds_importance:
            break
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = np.log1p(y.iloc[tr_idx])
        y_va = np.log1p(y.iloc[va_idx])
        sw_tr = sample_weight[tr_idx] if sample_weight is not None else None
        model = LGBMRegressor(
            n_estimators=600,
            learning_rate=0.05,
            max_depth=7,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=seed + fold,
            verbose=-1,
        )
        model.fit(
            X_tr,
            y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
        )
        acc += model.feature_importances_
        n_used += 1
    acc /= max(n_used, 1)
    order = np.argsort(acc)[::-1][:top_n]
    return [cols[i] for i in order]


feature_cols = select_top_features_lgb(
    train, y, groups, feature_cols, TOP_K_FEATURES, sample_weight=sw, n_folds_importance=3
)
X = train[feature_cols]
X_test = test[feature_cols]

decomp_feature_cols = [c for c in feature_cols if c not in DECOMP_DROP]
X_decomp = train[decomp_feature_cols]
X_test_decomp = test[decomp_feature_cols]

print(f"선택 후 feature 수: {len(feature_cols)} | 분해 전용: {len(decomp_feature_cols)}")
print(X.shape)
print(y.shape)
print(X_test.shape)
print(groups.shape)

(250000, 155)
(250000,)
(50000, 155)
(250000,)


### LightGBM 함수

In [48]:
def run_lgb_cv(X, y, X_test, groups, use_log_target=False, params=None, n_splits=5, sample_weight=None):
    if params is None:
        params = {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 7,
            'num_leaves': 63,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': 42,
            'verbose': -1,
        }

    kf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── LightGBM Fold {fold + 1} ──")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[valid_idx]
        y_tr = y.iloc[train_idx]
        y_val = y.iloc[valid_idx]

        if use_log_target:
            y_tr_fit = np.log1p(y_tr)
            y_val_fit = np.log1p(y_val)
        else:
            y_tr_fit = y_tr
            y_val_fit = y_val

        sw_tr = sample_weight[train_idx] if sample_weight is not None else None

        model = LGBMRegressor(**params)

        model.fit(
            X_tr, y_tr_fit,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val_fit)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )

        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        if use_log_target:
            val_pred = np.expm1(val_pred)
            test_pred = np.expm1(test_pred)

        val_pred = np.clip(val_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)

        oof_preds[valid_idx] = val_pred
        test_preds += test_pred / n_splits

    mae = mean_absolute_error(y, oof_preds)
    return oof_preds, test_preds, mae

### CatBoost 함수

In [49]:
def run_cat_cv(X, y, X_test, groups, use_log_target=False, params=None, n_splits=5, sample_weight=None):
    if params is None:
        params = {
            'iterations': 1000,
            'learning_rate': 0.05,
            'depth': 8,
            'loss_function': 'MAE',
            'eval_metric': 'MAE',
            'random_seed': 42,
            'verbose': 100
        }

    kf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── CatBoost Fold {fold + 1} ──")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[valid_idx]
        y_tr = y.iloc[train_idx]
        y_val = y.iloc[valid_idx]

        if use_log_target:
            y_tr_fit = np.log1p(y_tr)
            y_val_fit = np.log1p(y_val)
        else:
            y_tr_fit = y_tr
            y_val_fit = y_val

        sw_tr = sample_weight[train_idx] if sample_weight is not None else None

        model = CatBoostRegressor(**params)

        fit_kw = dict(
            eval_set=(X_val, y_val_fit),
            use_best_model=True,
            early_stopping_rounds=50,
        )
        if sw_tr is not None:
            fit_kw['sample_weight'] = sw_tr
        model.fit(X_tr, y_tr_fit, **fit_kw)

        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        if use_log_target:
            val_pred = np.expm1(val_pred)
            test_pred = np.expm1(test_pred)

        val_pred = np.clip(val_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)

        oof_preds[valid_idx] = val_pred
        test_preds += test_pred / n_splits

    mae = mean_absolute_error(y, oof_preds)
    return oof_preds, test_preds, mae

In [ ]:
def run_decomposition_cv(X, y_avg, y_total, y_order, X_test, groups, sample_weight=None, n_splits=5, seed=42):
    """총 지연(proxy)과 출고량(proxy)을 각각 예측 후 avg = total / order 로 복원."""
    params = {
        'n_estimators': 900,
        'learning_rate': 0.05,
        'max_depth': 7,
        'num_leaves': 63,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': seed,
        'verbose': -1,
    }
    kf = GroupKFold(n_splits=n_splits)
    oof_total = np.zeros(len(X))
    oof_order = np.zeros(len(X))
    test_total = np.zeros(len(X_test))
    test_order = np.zeros(len(X_test))

    y_total = pd.Series(y_total).reset_index(drop=True)
    y_order = pd.Series(y_order).reset_index(drop=True)
    y_avg = pd.Series(y_avg).reset_index(drop=True)

    for fold, (tr, va) in enumerate(kf.split(X, y_avg, groups=groups)):
        print(f"── Decomposition Fold {fold + 1} ──")
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        sw_tr = sample_weight[tr] if sample_weight is not None else None

        yt_tr = np.log1p(y_total.iloc[tr])
        yt_va = np.log1p(y_total.iloc[va])
        m_t = LGBMRegressor(**{**params, 'random_state': seed + fold})
        m_t.fit(
            X_tr, yt_tr, sample_weight=sw_tr,
            eval_set=[(X_va, yt_va)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )
        pr_t_va = np.expm1(m_t.predict(X_va))
        oof_total[va] = np.clip(pr_t_va, 0, None)

        yo_tr = np.log1p(y_order.iloc[tr])
        yo_va = np.log1p(y_order.iloc[va])
        m_o = LGBMRegressor(**{**params, 'random_state': seed + 100 + fold})
        m_o.fit(
            X_tr, yo_tr, sample_weight=sw_tr,
            eval_set=[(X_va, yo_va)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )
        pr_o_va = np.expm1(m_o.predict(X_va))
        oof_order[va] = np.clip(pr_o_va, 1e-6, None)

        test_total += np.clip(np.expm1(m_t.predict(X_test)), 0, None) / n_splits
        test_order += np.clip(np.expm1(m_o.predict(X_test)), 1e-6, None) / n_splits

    oof_avg = np.clip(oof_total / np.maximum(oof_order, 1.0), 0, None)
    test_avg = np.clip(test_total / np.maximum(test_order, 1.0), 0, None)
    mae = mean_absolute_error(y_avg, oof_avg)
    return oof_avg, test_avg, mae


def run_cat_quantile_blend_cv(
    X, y, X_test, groups,
    alphas=(0.5, 0.8, 0.9),
    blend_weights=(0.4, 0.35, 0.25),
    sample_weight=None,
    use_log_target=True,
    n_splits=5,
    seed=42,
):
    """CatBoost 분위수 회귀 앙상블 — 꼬리(tail) 쪽 가중."""
    kf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(X))
    test = np.zeros(len(X_test))
    w = np.array(blend_weights, dtype=float)
    w = w / w.sum()

    for fold, (tr, va) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── CatBoost Quantile blend Fold {fold + 1} ──")
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_val = y.iloc[tr], y.iloc[va]
        sw_tr = sample_weight[tr] if sample_weight is not None else None

        if use_log_target:
            y_tr_fit, y_val_fit = np.log1p(y_tr), np.log1p(y_val)
        else:
            y_tr_fit, y_val_fit = y_tr, y_val

        fold_va = np.zeros(len(va))
        fold_te = np.zeros(len(X_test))
        for a, wt in zip(alphas, w):
            params = {
                'iterations': 800,
                'learning_rate': 0.05,
                'depth': 8,
                'loss_function': f'Quantile:alpha={a}',
                'eval_metric': 'MAE',
                'random_seed': seed + fold + int(a * 100),
                'verbose': False,
            }
            model = CatBoostRegressor(**params)
            fit_kw = dict(
                eval_set=(X_va, y_val_fit),
                use_best_model=True,
                early_stopping_rounds=50,
            )
            if sw_tr is not None:
                fit_kw['sample_weight'] = sw_tr
            model.fit(X_tr, y_tr_fit, **fit_kw)
            pv = model.predict(X_va)
            pt = model.predict(X_test)
            if use_log_target:
                pv = np.expm1(pv)
                pt = np.expm1(pt)
            fold_va += wt * np.clip(pv, 0, None)
            fold_te += wt * np.clip(pt, 0, None)

        oof[va] = fold_va
        test += fold_te / n_splits

    mae = mean_absolute_error(y, oof)
    return oof, test, mae


def run_delay_level_moe_cv(
    X, y, y_level, X_test, groups, sample_weight=None, n_splits=5, seed=42,
    min_samples_per_level=80,
):
    """low/mid/high 분류 후 구간별 회귀를 확률 가중 혼합 (Mixture of Experts)."""
    y_level = pd.Series(y_level).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    kf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(X))
    test = np.zeros(len(X_test))

    reg_params = {
        'n_estimators': 700,
        'learning_rate': 0.05,
        'max_depth': 7,
        'num_leaves': 63,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': seed,
        'verbose': -1,
    }

    for fold, (tr, va) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── Delay-level MoE Fold {fold + 1} ──")
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_val = y.iloc[tr], y.iloc[va]
        lvl_tr = y_level.iloc[tr]
        sw_tr = sample_weight[tr] if sample_weight is not None else None

        clf = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=seed + fold,
            verbose=-1,
        )
        clf.fit(
            X_tr, lvl_tr,
            sample_weight=sw_tr,
            eval_set=[(X_va, y_level.iloc[va])],
            eval_metric='multi_logloss',
            callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
        )
        proba_va = clf.predict_proba(X_va)
        proba_te = clf.predict_proba(X_test)

        regs = []
        for k in range(3):
            mask = lvl_tr == k
            if mask.sum() < min_samples_per_level:
                m = LGBMRegressor(**{**reg_params, 'random_state': seed + 10 * k + fold})
                y_tr_fit = np.log1p(y_tr)
                y_va_fit = np.log1p(y_val)
                m.fit(
                    X_tr, y_tr_fit, sample_weight=sw_tr,
                    eval_set=[(X_va, y_va_fit)],
                    callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
                )
            else:
                m = LGBMRegressor(**{**reg_params, 'random_state': seed + 10 * k + fold})
                X_k = X_tr.loc[mask]
                y_k = y_tr.loc[mask]
                sw_k = sw_tr[mask.values] if sw_tr is not None else None
                y_k_fit = np.log1p(y_k)
                y_va_fit = np.log1p(y_val)
                m.fit(
                    X_k, y_k_fit, sample_weight=sw_k,
                    eval_set=[(X_va, y_va_fit)],
                    callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)],
                )
            regs.append(m)

        pred_va = sum(proba_va[:, k] * np.clip(np.expm1(regs[k].predict(X_va)), 0, None) for k in range(3))
        pred_te = sum(proba_te[:, k] * np.clip(np.expm1(regs[k].predict(X_test)), 0, None) for k in range(3))
        oof[va] = pred_va
        test += pred_te / n_splits

    mae = mean_absolute_error(y, oof)
    return oof, test, mae


def run_residual_lgb_cv(X, y, oof_base, X_test, test_base, groups, sample_weight=None, n_splits=5, seed=42):
    """1차 예측 잔차(residual)만 추가 학습 후 base + residual."""
    resid = y.values - oof_base
    sw_r = (1.0 + np.log1p(np.abs(resid)))
    if sample_weight is not None:
        sw_r = sw_r * sample_weight

    params = {
        'n_estimators': 600,
        'learning_rate': 0.04,
        'max_depth': 6,
        'num_leaves': 47,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.15,
        'reg_lambda': 0.15,
        'random_state': seed,
        'verbose': -1,
    }
    kf = GroupKFold(n_splits=n_splits)
    oof_res = np.zeros(len(X))
    test_res = np.zeros(len(X_test))

    for fold, (tr, va) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── Residual LGBM Fold {fold + 1} ──")
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        r_tr, r_va = resid[tr], resid[va]
        sw_tr = sw_r[tr]

        model = LGBMRegressor(**{**params, 'random_state': seed + fold})
        model.fit(
            X_tr, r_tr,
            sample_weight=sw_tr,
            eval_set=[(X_va, r_va)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )
        oof_res[va] = model.predict(X_va)
        test_res += model.predict(X_test) / n_splits

    oof_final = np.clip(oof_base + oof_res, 0, None)
    test_final = np.clip(test_base + test_res, 0, None)
    mae = mean_absolute_error(y, oof_final)
    return oof_final, test_final, oof_res, test_res, mae


def apply_layout_type_bias(oof_pred, test_pred, layout_train, layout_test, y_true):
    """layout_type별 OOF 잔차 평균으로 보정 (로컬 바이어스)."""
    df = pd.DataFrame({'lay': layout_train.values, 'e': y_true - oof_pred})
    bias = df.groupby('lay')['e'].mean()
    bt = layout_train.map(bias).fillna(0).values
    bte = layout_test.map(bias).fillna(0).values
    return np.clip(oof_pred + bt, 0, None), np.clip(test_pred + bte, 0, None)


### Two-Stage 함수

In [50]:
def run_two_stage_lgb_cv(
    X, y, X_test, groups,
    threshold=45,
    use_log_target=True,
    n_splits=5,
    clf_params=None,
    reg_params=None,
    high_reg_params=None
):
    if clf_params is None:
        clf_params = {
            'n_estimators': 600,
            'learning_rate': 0.05,
            'max_depth': 6,
            'num_leaves': 31,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': 42,
            'verbose': -1,
        }

    if reg_params is None:
        reg_params = {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 7,
            'num_leaves': 63,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': 42,
            'verbose': -1,
        }

    if high_reg_params is None:
        high_reg_params = {
            'n_estimators': 1200,
            'learning_rate': 0.04,
            'max_depth': 8,
            'num_leaves': 127,
            'subsample': 0.85,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.2,
            'random_state': 42,
            'verbose': -1,
        }

    kf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    y_cls = (y >= threshold).astype(int)

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── Two-Stage Fold {fold + 1} ──")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[valid_idx]

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[valid_idx]

        y_tr_cls = y_cls.iloc[train_idx]
        y_val_cls = y_cls.iloc[valid_idx]

        # 1) high_delay 분류 모델
        clf = LGBMClassifier(**clf_params)
        clf.fit(
            X_tr, y_tr_cls,
            eval_set=[(X_val, y_val_cls)],
            eval_metric='binary_logloss',
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )

        val_p_high = clf.predict_proba(X_val)[:, 1]
        test_p_high = clf.predict_proba(X_test)[:, 1]

        # 2) 일반 회귀 모델
        if use_log_target:
            y_tr_fit = np.log1p(y_tr)
            y_val_fit = np.log1p(y_val)
        else:
            y_tr_fit = y_tr
            y_val_fit = y_val

        reg_normal = LGBMRegressor(**reg_params)
        reg_normal.fit(
            X_tr, y_tr_fit,
            eval_set=[(X_val, y_val_fit)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )

        val_pred_normal = reg_normal.predict(X_val)
        test_pred_normal = reg_normal.predict(X_test)

        if use_log_target:
            val_pred_normal = np.expm1(val_pred_normal)
            test_pred_normal = np.expm1(test_pred_normal)

        val_pred_normal = np.clip(val_pred_normal, 0, None)
        test_pred_normal = np.clip(test_pred_normal, 0, None)

        # 3) 고지연 전용 회귀 모델
        high_mask = (y_tr >= threshold)

        # 고지연 샘플이 너무 적은 fold 대비 안전장치
        if high_mask.sum() >= 30:
            X_tr_high = X_tr.loc[high_mask]
            y_tr_high = y_tr.loc[high_mask]

            # validation도 같은 전체 X_val로 예측만 하면 됨
            if use_log_target:
                y_tr_high_fit = np.log1p(y_tr_high)
            else:
                y_tr_high_fit = y_tr_high

            reg_high = LGBMRegressor(**high_reg_params)
            reg_high.fit(
                X_tr_high, y_tr_high_fit,
                callbacks=[lgb.log_evaluation(100)],
            )

            val_pred_high = reg_high.predict(X_val)
            test_pred_high = reg_high.predict(X_test)

            if use_log_target:
                val_pred_high = np.expm1(val_pred_high)
                test_pred_high = np.expm1(test_pred_high)

            val_pred_high = np.clip(val_pred_high, 0, None)
            test_pred_high = np.clip(test_pred_high, 0, None)
        else:
            # 고지연 데이터가 적으면 일반 회귀값으로 대체
            val_pred_high = val_pred_normal.copy()
            test_pred_high = test_pred_normal.copy()

        # 4) 분류 확률로 혼합
        val_final = (1 - val_p_high) * val_pred_normal + val_p_high * val_pred_high
        test_final = (1 - test_p_high) * test_pred_normal + test_p_high * test_pred_high

        oof_preds[valid_idx] = val_final
        test_preds += test_final / n_splits

    mae = mean_absolute_error(y, oof_preds)
    return oof_preds, test_preds, mae

### 시퀀스 함수

In [51]:
def build_sequence_arrays(df, feature_cols, target_col=None, seq_len=5):
    X_seq = []
    y_seq = [] if target_col is not None else None
    row_indices = []

    for scenario_id, g in df.groupby('scenario_id', sort=False):
        g = g.reset_index()
        feat = g[feature_cols].values.astype(np.float32)

        if target_col is not None:
            target = g[target_col].values.astype(np.float32)

        for i in range(len(g)):
            start = max(0, i - seq_len + 1)
            seq = feat[start:i+1]

            # 앞쪽 부족한 길이는 0 padding
            if len(seq) < seq_len:
                pad = np.zeros((seq_len - len(seq), feat.shape[1]), dtype=np.float32)
                seq = np.vstack([pad, seq])

            X_seq.append(seq)
            row_indices.append(g.loc[i, 'index'])

            if target_col is not None:
                y_seq.append(target[i])

    X_seq = np.stack(X_seq)

    if target_col is not None:
        y_seq = np.array(y_seq, dtype=np.float32)
        return X_seq, y_seq, np.array(row_indices)
    else:
        return X_seq, np.array(row_indices)
    
class SeqDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

class GRURegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        pred = self.head(last).squeeze(1)
        return pred

### GRU 함수

In [52]:
def run_gru_cv(train_df, test_df, feature_cols, target_col, groups, seq_len=5, use_log_target=True,
               n_splits=5, batch_size=256, epochs=10, lr=1e-3, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    X_all, y_all, row_idx = build_sequence_arrays(train_df, feature_cols, target_col, seq_len=seq_len)
    X_test_all, test_row_idx = build_sequence_arrays(test_df, feature_cols, target_col=None, seq_len=seq_len)

    # row_idx 기준으로 원래 행 순서 복원
    order = np.argsort(row_idx)
    X_all = X_all[order]
    y_all = y_all[order]
    row_idx = row_idx[order]

    test_order = np.argsort(test_row_idx)
    X_test_all = X_test_all[test_order]
    test_row_idx = test_row_idx[test_order]

    oof_preds = np.zeros(len(train_df), dtype=np.float32)
    test_fold_preds = []

    kf = GroupKFold(n_splits=n_splits)

    dummy_X = train_df[feature_cols]
    y_series = train_df[target_col]

    for fold, (train_idx, valid_idx) in enumerate(kf.split(dummy_X, y_series, groups=groups)):
        print(f"── GRU Fold {fold + 1} ──")

        X_tr = X_all[train_idx]
        X_val = X_all[valid_idx]
        y_tr = y_all[train_idx]
        y_val = y_all[valid_idx]

        if use_log_target:
            y_tr_fit = np.log1p(y_tr)
            y_val_fit = np.log1p(y_val)
        else:
            y_tr_fit = y_tr
            y_val_fit = y_val

        train_ds = SeqDataset(X_tr, y_tr_fit)
        valid_ds = SeqDataset(X_val, y_val_fit)
        test_ds = SeqDataset(X_test_all)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

        model = GRURegressor(input_dim=X_tr.shape[2]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.L1Loss()

        best_val = float('inf')
        best_state = None
        patience = 3
        patience_cnt = 0

        for epoch in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                loss.backward()
                optimizer.step()

            model.eval()
            val_preds = []
            val_true = []
            with torch.no_grad():
                for xb, yb in valid_loader:
                    xb = xb.to(device)
                    pred = model(xb).cpu().numpy()
                    val_preds.append(pred)
                    val_true.append(yb.numpy())

            val_preds = np.concatenate(val_preds)
            val_true = np.concatenate(val_true)
            val_mae = np.mean(np.abs(val_preds - val_true))

            print(f"epoch={epoch+1}, val_mae={val_mae:.4f}")

            if val_mae < best_val:
                best_val = val_mae
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()

        with torch.no_grad():
            val_preds = []
            for xb, _ in valid_loader:
                xb = xb.to(device)
                pred = model(xb).cpu().numpy()
                val_preds.append(pred)
            val_preds = np.concatenate(val_preds)

            test_preds = []
            for xb in test_loader:
                xb = xb.to(device)
                pred = model(xb).cpu().numpy()
                test_preds.append(pred)
            test_preds = np.concatenate(test_preds)

        if use_log_target:
            val_preds = np.expm1(val_preds)
            test_preds = np.expm1(test_preds)

        val_preds = np.clip(val_preds, 0, None)
        test_preds = np.clip(test_preds, 0, None)

        oof_preds[valid_idx] = val_preds
        test_fold_preds.append(test_preds)

    test_preds_mean = np.mean(test_fold_preds, axis=0)
    mae = mean_absolute_error(train_df[target_col], oof_preds)

    return oof_preds, test_preds_mean, mae

### LightGBM 실행

In [55]:
# 통합 파이프라인: 타깃 분해 + CatBoost 분위수 + 가중 LightGBM + delay-level MoE + residual + layout 보정
y_total = train['_total_delay_proxy']
y_order = train['_order_vol_proxy']
y_level = train['_delay_level']

print('=== 1) 분해: total_delay_proxy / order_vol_proxy → avg 복원 ===')
dec_oof, dec_test, dec_mae = run_decomposition_cv(
    X_decomp, y, y_total, y_order, X_test_decomp, groups, sample_weight=sw, n_splits=5,
)
print(f'[Decomposition] OOF MAE: {dec_mae:.4f}')

print('=== 2) CatBoost Quantile (q50/q80/q90 가중) ===')
catq_oof, catq_test, catq_mae = run_cat_quantile_blend_cv(
    X, y, X_test, groups,
    alphas=(0.5, 0.8, 0.9),
    blend_weights=(0.4, 0.35, 0.25),
    sample_weight=sw,
    use_log_target=True,
    n_splits=5,
)
print(f'[CatBoost Quantile] OOF MAE: {catq_mae:.4f}')

print('=== 3) CatBoost MAE (스택/비교용) ===')
cat_oof, cat_test, cat_mae = run_cat_cv(
    X, y, X_test, groups, use_log_target=True, sample_weight=sw,
)
print(f'[CatBoost log1p+weight] OOF MAE: {cat_mae:.4f}')

print('=== 4) LightGBM log1p + sample_weight ===')
lgb_oof, lgb_test, lgb_mae = run_lgb_cv(
    X, y, X_test, groups, use_log_target=True, sample_weight=sw,
)
print(f'[LightGBM log1p+weight] OOF MAE: {lgb_mae:.4f}')

oof_raw, test_raw, mae_raw = run_lgb_cv(
    X, y, X_test, groups, use_log_target=False, sample_weight=sw,
)
oof_log, test_log, mae_log = lgb_oof, lgb_test, lgb_mae

print('=== 5) Delay-level MoE (low/mid/high) ===')
dl_oof, dl_test, dl_mae = run_delay_level_moe_cv(
    X, y, y_level, X_test, groups, sample_weight=sw, n_splits=5,
)
print(f'[Delay-level MoE] OOF MAE: {dl_mae:.4f}')

print('=== 6) 베이스 블렌드 + residual + layout_type 잔차 보정 ===')
w_d, w_c, w_l, w_dl = 0.22, 0.33, 0.28, 0.17
base_oof = w_d * dec_oof + w_c * catq_oof + w_l * lgb_oof + w_dl * dl_oof
base_test = w_d * dec_test + w_c * catq_test + w_l * lgb_test + w_dl * dl_test
print(f'[Base blend] OOF MAE: {mean_absolute_error(y, base_oof):.4f}')

adv_oof, adv_test, _, _, _ = run_residual_lgb_cv(
    X, y, base_oof, X_test, base_test, groups, sample_weight=sw, n_splits=5,
)
print(f'[After residual] OOF MAE: {mean_absolute_error(y, adv_oof):.4f}')

adv_oof, adv_test = apply_layout_type_bias(
    adv_oof, adv_test, train['layout_type'], test['layout_type'], y.values,
)
advanced_mae = mean_absolute_error(y, adv_oof)
print(f'[Advanced final] OOF MAE: {advanced_mae:.4f}')

── LightGBM Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 486.606
Early stopping, best iteration is:
[125]	valid_0's l2: 486.336
── LightGBM Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 478.872
Early stopping, best iteration is:
[69]	valid_0's l2: 477.6
── LightGBM Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 473.451
Early stopping, best iteration is:
[101]	valid_0's l2: 473.391
── LightGBM Fold 4 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 373.676
Early stopping, best iteration is:
[133]	valid_0's l2: 373.029
── LightGBM Fold 5 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 433.488
Early stopping, best iteration is:
[141]	valid_0's l2: 433.058
[원본 target] OOF MAE: 9.5547
── LightGBM Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.

### CatBoost 실행

In [56]:
# CatBoost는 위 통합 셀에서 이미 학습됨 (Quantile catq_* + MAE cat_*)
print('[CatBoost] cat_oof/cat_test = MAE+log1p+가중 | catq_* = 분위수 블렌드')

── CatBoost Fold 1 ──
0:	learn: 14.1793495	test: 14.2790960	best: 14.2790960 (0)	total: 45.3ms	remaining: 45.3s
100:	learn: 8.8660318	test: 9.2394412	best: 9.2394412 (100)	total: 4.25s	remaining: 37.8s
200:	learn: 8.5427318	test: 9.1916956	best: 9.1914115 (198)	total: 8.28s	remaining: 32.9s
300:	learn: 8.2599294	test: 9.1619586	best: 9.1619586 (300)	total: 12.3s	remaining: 28.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 9.158480247
bestIteration = 308

Shrink model to first 309 iterations.
── CatBoost Fold 2 ──
0:	learn: 14.2147044	test: 14.1217366	best: 14.1217366 (0)	total: 42.5ms	remaining: 42.5s
100:	learn: 8.8760562	test: 9.0733504	best: 9.0733504 (100)	total: 4.16s	remaining: 37s
200:	learn: 8.5817020	test: 9.0345191	best: 9.0345191 (200)	total: 8.08s	remaining: 32.1s
300:	learn: 8.3174372	test: 9.0158163	best: 9.0116126 (276)	total: 12s	remaining: 28s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 9.005547002
bestIteration = 312

Shrink

### 평균 앙상블 실행

In [57]:
ensemble_oof = (lgb_oof + cat_oof) / 2
ensemble_test = (lgb_test + cat_test) / 2
ensemble_mae = mean_absolute_error(y, ensemble_oof)

# 분위수·분해·MoE를 반영한 넓은 앙상블 (비교용)
wide_oof = (dec_oof + catq_oof + lgb_oof + dl_oof) / 4
wide_test = (dec_test + catq_test + lgb_test + dl_test) / 4
wide_mae = mean_absolute_error(y, wide_oof)

print(f'[LGB+Cat MAE 앙상블] OOF MAE: {ensemble_mae:.4f}')
print(f'[Wide 4-way] OOF MAE: {wide_mae:.4f}')

[앙상블] OOF MAE: 9.0958


### Two-stage 실행

In [58]:
results_two_stage = {}

for th in [30, 45, 60]:
    print(f"\n===== threshold = {th} =====")
    oof_tmp, test_tmp, mae_tmp = run_two_stage_lgb_cv(
        X=X,
        y=y,
        X_test=X_test,
        groups=groups,
        threshold=th,
        use_log_target=True
    )
    results_two_stage[th] = {
        'oof': oof_tmp,
        'test': test_tmp,
        'mae': mae_tmp
    }
    print(f"[Two-Stage threshold={th}] OOF MAE: {mae_tmp:.4f}")

best_th = min(results_two_stage, key=lambda k: results_two_stage[k]['mae'])
best_two_stage_oof = results_two_stage[best_th]['oof']
best_two_stage_test = results_two_stage[best_th]['test']
best_two_stage_mae = results_two_stage[best_th]['mae']

print(f"\n최종 선택 threshold: {best_th}")
print(f"최종 Two-Stage OOF MAE: {best_two_stage_mae:.4f}")


===== threshold = 30 =====
── Two-Stage Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.273525
Early stopping, best iteration is:
[93]	valid_0's binary_logloss: 0.273488
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.439184
[200]	valid_0's l2: 0.436889
Early stopping, best iteration is:
[169]	valid_0's l2: 0.436677
── Two-Stage Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.269239
[200]	valid_0's binary_logloss: 0.268931
Early stopping, best iteration is:
[163]	valid_0's binary_logloss: 0.268487
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.448247
[200]	valid_0's l2: 0.446143
Early stopping, best iteration is:
[162]	valid_0's l2: 0.44582
── Two-Stage Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.290187
Early stopping, best iteration is:
[148]

### GRU 실행

In [59]:
gru_oof, gru_test, gru_mae = run_gru_cv(
    train_df=train,
    test_df=test,
    feature_cols=seq_feature_cols,
    target_col=TARGET,
    groups=train['scenario_id'],
    seq_len=5,
    use_log_target=True,
    epochs=10
)

print(f"[GRU] OOF MAE: {gru_mae:.4f}")

── GRU Fold 1 ──
epoch=1, val_mae=0.5810
epoch=2, val_mae=0.5638
epoch=3, val_mae=0.5556
epoch=4, val_mae=0.5382
epoch=5, val_mae=0.5350
epoch=6, val_mae=0.5300
epoch=7, val_mae=0.5418
epoch=8, val_mae=0.5287
epoch=9, val_mae=0.5269
epoch=10, val_mae=0.5245
── GRU Fold 2 ──
epoch=1, val_mae=0.5807
epoch=2, val_mae=0.5654
epoch=3, val_mae=0.5550
epoch=4, val_mae=0.5440
epoch=5, val_mae=0.5346
epoch=6, val_mae=0.5316
epoch=7, val_mae=0.5454
epoch=8, val_mae=0.5382
epoch=9, val_mae=0.5354
── GRU Fold 3 ──
epoch=1, val_mae=0.5700
epoch=2, val_mae=0.5637
epoch=3, val_mae=0.5613
epoch=4, val_mae=0.5642
epoch=5, val_mae=0.5526
epoch=6, val_mae=0.5498
epoch=7, val_mae=0.5684
epoch=8, val_mae=0.5523
epoch=9, val_mae=0.5483
epoch=10, val_mae=0.5453
── GRU Fold 4 ──
epoch=1, val_mae=0.5695
epoch=2, val_mae=0.5391
epoch=3, val_mae=0.5324
epoch=4, val_mae=0.5285
epoch=5, val_mae=0.5238
epoch=6, val_mae=0.5188
epoch=7, val_mae=0.5187
epoch=8, val_mae=0.5238
epoch=9, val_mae=0.5231
epoch=10, val_mae=

### Stacking 실행

In [61]:
from sklearn.linear_model import Ridge

stack_train = pd.DataFrame({
    'lgb_best': lgb_oof,
    'cat_best': cat_oof,
    'cat_quantile': catq_oof,
    'decomp': dec_oof,
    'delay_moe': dl_oof,
    'advanced': adv_oof,
    'lgb_raw': oof_raw,
    'lgb_log': oof_log,
    'two_stage': best_two_stage_oof,
    'gru': gru_oof,
})

stack_test = pd.DataFrame({
    'lgb_best': lgb_test,
    'cat_best': cat_test,
    'cat_quantile': catq_test,
    'decomp': dec_test,
    'delay_moe': dl_test,
    'advanced': adv_test,
    'lgb_raw': test_raw,
    'lgb_log': test_log,
    'two_stage': best_two_stage_test,
    'gru': gru_test,
})

def run_stack_cv(stack_train, y, stack_test, groups, alpha=1.0, n_splits=5):
    kf = GroupKFold(n_splits=n_splits)

    oof_preds = np.zeros(len(stack_train))
    test_preds = np.zeros(len(stack_test))

    for fold, (train_idx, valid_idx) in enumerate(kf.split(stack_train, y, groups=groups)):
        print(f"── Stacking Fold {fold + 1} ──")

        X_tr = stack_train.iloc[train_idx]
        X_val = stack_train.iloc[valid_idx]
        y_tr = y.iloc[train_idx]

        model = Ridge(alpha=alpha)
        model.fit(X_tr, y_tr)

        val_pred = model.predict(X_val)
        test_pred = model.predict(stack_test)

        val_pred = np.clip(val_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)

        oof_preds[valid_idx] = val_pred
        test_preds += test_pred / n_splits

    mae = mean_absolute_error(y, oof_preds)
    return oof_preds, test_preds, mae

stack_oof, stack_test_pred, stack_mae = run_stack_cv(
    stack_train=stack_train,
    y=train[TARGET],
    stack_test=stack_test,
    groups=train['scenario_id'],
    alpha=1.0
)

print(f"[Stacking-Ridge] OOF MAE: {stack_mae:.4f}")

── Stacking Fold 1 ──
── Stacking Fold 2 ──
── Stacking Fold 3 ──
── Stacking Fold 4 ──
── Stacking Fold 5 ──
[Stacking-Ridge] OOF MAE: 9.4628


### 최종 비교 및 선택

In [62]:
results = {
    'lightgbm': lgb_mae,
    'catboost': cat_mae,
    'catboost_quantile': catq_mae,
    'decomposition': dec_mae,
    'delay_moe': dl_mae,
    'advanced_pipeline': advanced_mae,
    'ensemble': ensemble_mae,
    'wide_4way': wide_mae,
    'two_stage_lgb': best_two_stage_mae,
    'gru': gru_mae,
    'stacking_ridge': stack_mae,
}

best_model_name = min(results, key=results.get)
best_score = results[best_model_name]

print('모델별 OOF MAE:', results)
print(f'최종 선택: {best_model_name}, OOF MAE: {best_score:.4f}')

if best_model_name == 'lightgbm':
    final_test_preds = lgb_test
elif best_model_name == 'catboost':
    final_test_preds = cat_test
elif best_model_name == 'catboost_quantile':
    final_test_preds = catq_test
elif best_model_name == 'decomposition':
    final_test_preds = dec_test
elif best_model_name == 'delay_moe':
    final_test_preds = dl_test
elif best_model_name == 'advanced_pipeline':
    final_test_preds = adv_test
elif best_model_name == 'ensemble':
    final_test_preds = ensemble_test
elif best_model_name == 'wide_4way':
    final_test_preds = wide_test
elif best_model_name == 'two_stage_lgb':
    final_test_preds = best_two_stage_test
elif best_model_name == 'gru':
    final_test_preds = gru_test
else:
    final_test_preds = stack_test_pred

모델별 OOF MAE: {'lightgbm': 9.554683942360622, 'catboost': 9.020416441128452, 'ensemble': 9.095790199462764, 'two_stage_lgb': 9.291364716585521, 'gru': 9.657242043755565, 'stacking_ridge': 9.462792879927287}
최종 선택: catboost, OOF MAE: 9.0204


## 제출 파일 저장 및 확인

In [63]:
submission = pd.DataFrame({
    'ID': test['ID'],
    TARGET: final_test_preds
})

submission.to_csv('./submission.csv', index=False)
print("submission.csv 저장 완료.")

check = pd.read_csv('./submission.csv')
print(check.shape)
print(check.head())
print(check[TARGET].min(), check[TARGET].max())

submission.csv 저장 완료.
(50000, 2)
            ID  avg_delay_minutes_next_30m
0  TEST_015375                   18.523183
1  TEST_015376                   30.662146
2  TEST_015377                   32.934162
3  TEST_015378                   33.976461
4  TEST_015379                   34.871764
0.0735875772862193 53.63097614726319


In [64]:
check = pd.read_csv('./submission.csv')
print(check.shape)
print(check.head())
print(check['avg_delay_minutes_next_30m'].min(), check['avg_delay_minutes_next_30m'].max())

(50000, 2)
            ID  avg_delay_minutes_next_30m
0  TEST_015375                   18.523183
1  TEST_015376                   30.662146
2  TEST_015377                   32.934162
3  TEST_015378                   33.976461
4  TEST_015379                   34.871764
0.0735875772862193 53.63097614726319


In [65]:
print("index 동일:", submission.index.equals(check.index))
print("ID 동일:", (submission['ID'] == check['ID']).all())
print("dtype 비교:")
print(submission.dtypes)
print(check.dtypes)

pred_col = 'avg_delay_minutes_next_30m'

diff = np.abs(submission[pred_col].values - check[pred_col].values)
print("allclose:", np.allclose(submission[pred_col].values, check[pred_col].values))
print("최대 차이:", diff.max())
print("차이 있는 행 수:", (diff > 0).sum())

np.allclose(submission[pred_col], check[pred_col])

index 동일: True
ID 동일: True
dtype 비교:
ID                             object
avg_delay_minutes_next_30m    float64
dtype: object
ID                             object
avg_delay_minutes_next_30m    float64
dtype: object
allclose: True
최대 차이: 7.105427357601002e-15
차이 있는 행 수: 6072


True

## 3. 피처 및 타겟 설정

new_ratio_cols = [
    'order_per_robot',
    'charge_pressure',
    'congestion_per_robot',
    'orders_per_pack_station',
    'active_per_aisle_width',
    'congestion_per_aisle',
    'charging_per_robot',
    'low_battery_pressure',
    'pack_pressure',
    'intersection_pressure',
    'robot_density',
    'charger_per_robot',
    'emergency_complexity'
]

print(train[new_ratio_cols].head())
print(train[new_ratio_cols].isnull().sum())

two_stage_oof, two_stage_test, two_stage_mae = run_two_stage_lgb_cv(
    X=X,
    y=y,
    X_test=X_test,
    groups=groups,
    threshold=45,   # 먼저 45부터
    use_log_target=True
)

print(f"[Two-Stage threshold=45] OOF MAE: {two_stage_mae:.4f}")
print(f"prediction min/max: {two_stage_test.min():.4f}, {two_stage_test.max():.4f}")

**(참고)** LightGBM / CatBoost 학습은 위 **통합 파이프라인** 셀에서 수행합니다. 이 블록은 예전 실험용이며, 그대로 두었습니다. 재실행하면 `lgb_oof` 등이 덮어씌워지므로 사용하지 마세요.

**(참고)** CatBoost 비교 실험은 통합 셀에서 `cat_oof`(MAE)와 `catq_oof`(분위수)로 대체되었습니다.

## 6. 제출 파일 생성